# 自建长期记忆全流程教学：抽取 → 合并 → 存储 → 检索 → 使用

对照工业架构图，用 **`SelfBuiltLongTermMemory_improve.py`** 分模块演示（向量嵌入默认 **阿里云 DashScope**，不下载 BGE）。

| 模块 | 本课做什么 | 对应代码 |
|------|------------|----------|
| **1. 抽取** | LLM 把自然语言压成 `summary / key_points / importance` | `MemoryCompressor.extract_memory` |
| **2. 合并** | 两条相似记忆合成一条 | `MemoryCompressor.merge_memories` |
| **3. 存储** | 写入管道：去重 → 合并 → Chroma 索引 | `ingest_long_term` / `MemoryWritePipeline` |
| **4. 检索** | 向量 + BM25 + RRF，再按时间/重要性重排 | `search_memories` |
| **5. 使用** | 短期 Checkpoint 模拟 + 长期记忆拼进 Prompt（可选调 LLM） | `build_prompt` + `ChatOpenAI` |

**建议:从上到下 **Run All**。

> 第 1、2 模块会调用 LLM（记忆抽取/合并）；第 3～5 模块会调用 **DashScope Embedding**。请提前配置 `.env`。

## 0. 环境与路径

```bash
pip install langchain-core langchain-openai langchain-community chromadb rank-bm25 openai python-dotenv httpx numpy ipywidgets
```

`.env` 示例（项目根目录）：

```env
# LLM（内网 ModelFactory / OpenRouter 均可）
OPENAI_API_KEY=...
OPENAI_BASE_URL=https://modelfactory.example.com/api
OPENAI_MODEL=gpt-oss-120b
OPENAI_SSL_VERIFY=false

# 文本向量（阿里云百炼 OpenAI 兼容）
DASHSCOPE_API_KEY=sk-...
DASHSCOPE_EMBEDDING_MODEL=text-embedding-v3
DASHSCOPE_EMBEDDING_BASE_URL=https://dashscope.aliyuncs.com/compatible-mode/v1
```

`.env` 里只能写 `KEY=VALUE` 一行一项。

In [15]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "SelfBuiltLongTermMemory_improve.py").exists():
    # 本地调试目录
    ROOT = Path(r"d:\myproject\agent-systems-book")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if sys.platform == "win32":
    for _s in (sys.stdout, sys.stderr):
        try:
            _s.reconfigure(encoding="utf-8")
        except Exception:
            pass

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

print("项目根目录:", ROOT)
print("DASHSCOPE_API_KEY 已配置:", bool(os.getenv("DASHSCOPE_API_KEY", "").strip()))
print("LLM API 已配置:", bool((os.getenv("OPENAI_API_KEY") or os.getenv("OPENROUTER_API_KEY") or "").strip()))

项目根目录: D:\myproject\enterprise_bench_langchain
DASHSCOPE_API_KEY 已配置: True
LLM API 已配置: True


## 架构一览（写入 vs 读取）

```
对话/事实文本 ──► [抽取] ──► [去重?] ──► [合并?] ──► embed ──► Chroma
                                                      ▲
用户新问题 ──► 向量检索 + BM25/RRF ──► 重排 ──► 与短期对话拼 Prompt ──► LLM 回答
```
  模拟 LangGraph Checkpoint 中的 messages（只追加、不向量检索）
- **短期**：`ThreadShortTermMemory` 模拟 LangGraph **Checkpoint** 的 `messages`（不走向量检索）。
- **长期**：Chroma + 元数据（`user_id`、`memory_type`、`importance` 等）（向量检索）。

---
## 模块 1 · 记忆抽取（Extract）

输入：一句用户侧「可入库」的自然语言。  
输出：结构化 JSON → Python 字典（`summary`、`key_points`、`importance`）。

这一步 **尚未写入向量库**，便于单独观察 LLM 抽取效果。

In [16]:
from SelfBuiltLongTermMemory_improve import SelfBuiltLongTermMemoryImproved
from EmbeddingFactory import MemoryCompressor

# 只借用默认 LLM 配置，暂不创建 Chroma
_ltm_stub = SelfBuiltLongTermMemoryImproved.__new__(SelfBuiltLongTermMemoryImproved)
llm = SelfBuiltLongTermMemoryImproved._default_llm(_ltm_stub)
compressor = MemoryCompressor(llm)

RAW_UTTERANCE = "用户喜欢喝美式咖啡，不加糖不加奶"
extracted = compressor.extract_memory(RAW_UTTERANCE)

print("【原始输入】", RAW_UTTERANCE)
print("\n【抽取结果】")
print(json.dumps(
    {k: extracted[k] for k in ("summary", "key_points", "importance")},
    ensure_ascii=False,
    indent=2,
))

【原始输入】 用户喜欢喝美式咖啡，不加糖不加奶

【抽取结果】
{
  "summary": "用户喜欢喝不加糖不加奶的美式咖啡",
  "key_points": [
    "喜欢美式咖啡",
    "不加糖",
    "不加奶"
  ],
  "importance": 0.9
}


---
## 模块 2 · 记忆合并（Merge）

当两条记忆语义相近时，工业界通常 **合并为一条** 而不是无限新增。  
这里手动构造「旧记忆 + 新抽取」，调用 `merge_memories` 观察合并后的摘要与要点。

In [17]:
old_memory = {
    "id": "mem-old-001",
    "content": "用户喜欢喝美式咖啡，不加糖不加奶",
    "summary": "用户偏好无糖无奶美式咖啡",
    "key_points": ["喜欢美式", "不加糖", "不加奶"],
    "importance": 0.8,
}

new_memory = compressor.extract_memory("用户说以后咖啡都要美式，绝对不要糖奶")
merged = compressor.merge_memories([old_memory, new_memory])

print("【合并前】")
print("  旧:", old_memory["summary"])
print("  新:", new_memory["summary"])
print("\n【合并后】")
print(json.dumps(
    {k: merged[k] for k in ("summary", "key_points", "importance")},
    ensure_ascii=False,
    indent=2,
))

【合并前】
  旧: 用户偏好无糖无奶美式咖啡
  新: 用户要求以后所有咖啡只能是美式，绝对不加糖和奶

【合并后】
{
  "summary": "用户坚持只喝不加糖奶的美式咖啡，并表示以后所有咖啡都要如此。",
  "key_points": [
    "喜欢美式咖啡",
    "不加糖和奶",
    "今后所有咖啡都要求是美式且不加糖奶"
  ],
  "importance": 0.9
}


---
## 模块 3 · 存储（Store）

完整写入管道：**抽取 → 向量近邻去重 →（命中则）合并 → 写入 Chroma → 刷新 BM25 索引**。

- 使用独立目录 `./chroma_notebook_test`，避免与命令行 demo 互相覆盖。
- 第 3 条与第 1 条语义接近，预期 **库内仍为 2 条**（触发合并而非新增）。

In [19]:
import shutil

USER_ID = "student_demo"
CHROMA_DIR = ROOT / "chroma_notebook_2"

# 教学演示：每次 Run 可清空重建（若只想增量写入，注释掉下面两行）
# if CHROMA_DIR.exists():
#     shutil.rmtree(CHROMA_DIR)

ltm = SelfBuiltLongTermMemoryImproved(
    user_id=USER_ID,
    embedding_model="dashscope",
    persist_directory=str(CHROMA_DIR),
    collection_name="notebook_long_term_v1",
    similarity_threshold=0.0,  # 教学：放宽过滤，便于看到检索结果
)

samples = [
    ("用户喜欢喝美式咖啡，不加糖不加奶", "preference", 0.8),
    ("用户住在北京朝阳区", "fact", 0.5),
    ("用户偏好美式咖啡，不要糖奶", "preference", None),  # 近似重复 → 合并
]

print("=== 写入管道 ingest_long_term ===")
for text, mtype, imp in samples:
    mid = ltm.ingest_long_term(text, memory_type=mtype, importance=imp)
    print(f"  写入/更新 id={mid[:8]}… | type={mtype} | text={text[:24]}…")

print(f"\n库内文档条数: {len(ltm.documents)}（期望 2）")

=== 写入管道 ingest_long_term ===
  写入/更新 id=54fcb9ee… | type=preference | text=用户喜欢喝美式咖啡，不加糖不加奶…
  写入/更新 id=679afb17… | type=fact | text=用户住在北京朝阳区…
  写入/更新 id=54fcb9ee… | type=preference | text=用户偏好美式咖啡，不要糖奶…

库内文档条数: 2（期望 2）


In [22]:
print("=== Chroma 中当前记忆（摘要 + 元数据）===")
for i, (doc_id, doc, meta) in enumerate(zip(ltm.ids, ltm.documents, ltm.metadatas), 1):
    print(f"\n--- [{i}] id={doc_id[:8]}… ---")
    print("  summary:", meta.get("summary", ""))
    print("  memory_type:", meta.get("memory_type"))
    print("  importance:", meta.get("importance"))
    print("  content:", doc[:80] + ("…" if len(doc) > 80 else ""))

=== Chroma 中当前记忆（摘要 + 元数据）===

--- [1] id=54fcb9ee… ---
  summary: 用户偏好美式咖啡且不加糖和奶。
  memory_type: preference
  importance: 0.5
  content: 喜欢美式咖啡
不加糖
不加奶

--- [2] id=679afb17… ---
  summary: 用户住在北京朝阳区
  memory_type: fact
  importance: 0.5
  content: 用户住在北京朝阳区


---
## 模块 4 · 检索（Retrieve）

1. **密集检索**：Chroma 用 query 的 embedding 找近邻。  
2. **稀疏检索**：BM25 在内存文档列表上打分。  
3. **RRF 融合** + **时间衰减 / 重要性** 重排 → `search_memories` 返回 top-k。

可修改下方 `QUERY` 观察命中变化。

In [23]:
QUERY = "你还记得我喜欢喝什么咖啡吗？"

# 4.1 仅看 Chroma 向量检索（密集）
dense = ltm.collection.query(
    query_texts=[QUERY],
    n_results=5,
    where={"user_id": USER_ID},
)
print("【 向量检索】distance 越小越近")
for mid, dist, meta in zip(
    dense["ids"][0],
    dense["distances"][0],
    dense["metadatas"][0],
):
    sim = ltm._distance_to_similarity(dist)
    print(f"  - {meta.get('summary', '')[:40]} | dist={dist:.4f} sim≈{sim:.4f}")

【 向量检索】distance 越小越近
  - 用户偏好美式咖啡且不加糖和奶。 | dist=0.3617 sim≈0.8191
  - 用户住在北京朝阳区 | dist=0.5693 sim≈0.7154


In [24]:
# 4.2 完整混合检索 + 重排
hits = ltm.search_memories(QUERY, top_k=5)

print("【混合检索 search_memories】")
if not hits:
    print("  （无命中，可检查 similarity_threshold 或是否已写入）")
else:
    for h in hits:
        print(
            f"  - [{h['memory_type']}] {h['summary']} | score={h['score']:.4f} | imp={h['importance']}"
        )

【混合检索 search_memories】
  - [preference] 用户偏好美式咖啡且不加糖和奶。 | score=0.1617 | imp=0.5
  - [fact] 用户住在北京朝阳区 | score=0.1615 | imp=0.5


---
## 模块 5 · 使用（Use）

把 **短期对话（Checkpoint 模拟）** 与 **长期检索结果** 填入模板，得到可直接送给 LLM 的 Prompt。

最后一格可选 `RUN_LLM = True` 真调模型生成回答（会消耗 token）。

In [25]:
from SelfBuiltLongTermMemory_improve import MEMORY_PROMPT_TEMPLATE

THREAD_ID = "thread_notebook_001"

ltm.record_turn(THREAD_ID, "你好，我是小明", "你好小明，有什么可以帮你？")

prompt = ltm.build_prompt(THREAD_ID, QUERY, long_term_hits=hits)

print("【Prompt 模板占位符】recent_dialogue / memory_context / query\n")
print(prompt)
print("\n--- 模板原文（未填充）---\n")
print(MEMORY_PROMPT_TEMPLATE)

【Prompt 模板占位符】recent_dialogue / memory_context / query

你是一个有记忆的智能助手。请结合【最近对话】与【相关长期记忆】回答用户问题。

【最近对话】（来自 Checkpoint / 本线程短期缓冲，按时间顺序）
user: 你好，我是小明
assistant: 你好小明，有什么可以帮你？

【相关长期记忆】（向量检索 + 重排后的 top-k）
- [preference] 用户偏好美式咖啡且不加糖和奶。 (score=0.1617)
- [fact] 用户住在北京朝阳区 (score=0.1615)

【当前问题】
你还记得我喜欢喝什么咖啡吗？


--- 模板原文（未填充）---

你是一个有记忆的智能助手。请结合【最近对话】与【相关长期记忆】回答用户问题。

【最近对话】（来自 Checkpoint / 本线程短期缓冲，按时间顺序）
{recent_dialogue}

【相关长期记忆】（向量检索 + 重排后的 top-k）
{memory_context}

【当前问题】
{query}



In [26]:
# 可选：用同一套 LLM 配置生成最终回答
RUN_LLM = True  # 课堂演示可先 False，只展示 Prompt；课后改 True

if RUN_LLM:
    resp = llm.invoke(prompt)
    text = resp.content if isinstance(resp.content, str) else str(resp.content)
    print("【模型回答】\n", text)
else:
    print("RUN_LLM=False：本格跳过模型调用。将 RUN_LLM 改为 True 即可体验完整闭环。")

【模型回答】
 当然记得！你喜欢喝美式咖啡，而且不加糖也不加奶。有什么想要点的，或者需要我帮你查找附近的咖啡店吗？


---
## 小结

1. **抽取**：非结构化对话 → 结构化记忆字段。  
2. **合并**：相似记忆归并，控制库膨胀。  
3. **存储**：向量库 + 元数据 + BM25 倒排索引同步刷新。  
4. **检索**：语义 + 关键词混合，比单一向量更稳。  
5. **使用**：短期上下文 + 长期事实一起进 Prompt，才是「有记忆的 Agent」。

